In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA

In [2]:
df_train = pd.read_csv("../data_train_frequency.csv")
df_train.drop(columns=["Unnamed: 0"],inplace=True)

In [3]:
x_train = df_train.iloc[:,1:].values
y_train = df_train.iloc[:,0].values

In [4]:
df_test = pd.read_csv("../data_test_frequency.csv")
df_test.drop(columns=["Unnamed: 0"],inplace=True)

In [5]:
x_test = df_test.iloc[:,1:].values
y_test = df_test.iloc[:,0].values

In [6]:
var_pca = [i/100 for i in range(90,100)]
res_timerun = []

In [ ]:
for var in var_pca:
    scale = MinMaxScaler()
    x_train_sc = scale.fit_transform(x_train)

    pca = PCA(n_components=var)
    x_train_pca = pca.fit_transform(x_train_sc)
    d_train = x_train_pca.shape[1]

    x_test_sc = scale.transform(x_test)
    x_test_pca = pca.transform(x_test_sc)

    import time
    start_time = time.perf_counter()

    from xgboost import XGBClassifier
    from sklearn.model_selection import GridSearchCV
    model = XGBClassifier()
    params = {
        'n_estimators': [500,800 ,900,950],
        'learning_rate': [0.05,0.1],
        'max_depth': [3,4],
        'min_child_weight':[1],
        'gamma':[0],
    }
    grid_search = GridSearchCV(estimator=model, param_grid=params, cv=3, verbose=5, return_train_score=True,refit=True)
    grid_model = grid_search.fit(x_train_pca,y_train)

    end_time = time.perf_counter()
    res_etime = end_time - start_time
    res_timerun.append(res_etime)
    
    result_test = grid_model.predict(x_test_pca)
    result_train = grid_model.predict_proba(x_train_pca)
    from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay,multilabel_confusion_matrix,f1_score,precision_score,accuracy_score,recall_score,precision_recall_fscore_support
    def evaluation_test(y,y_pred, time):
        cm = confusion_matrix(y,y_pred)
        disp = ConfusionMatrixDisplay(cm,display_labels=['AFIB','SB','SR','GSVT'])
        disp.plot()
        plt.show()
        n_classes = len(cm)
        class_counts = np.sum(cm, axis=1)
        result = []
        specificities = []
        weights = class_counts / np.sum(class_counts) 
        for c in range(n_classes):
            tp = cm[c,c]
            fp = sum(cm[:,c]) - cm[c,c]
            fn = sum(cm[c,:]) - cm[c,c]
            tn = sum(np.delete(sum(cm)-cm[c,:],c))
            acc = (tp+tn) / (tp+fn+tn+fp)
            recall = tp/(tp+fn)
            precision = tp/(tp+fp)
            specificity = tn/(tn+fp)
            specificities.append(specificity)
            f1_score = 2*((precision*recall)/(precision+recall))
            if c+1 == 1:
                Rhythm = 'AFIB'
            elif c+1 == 2:
                Rhythm = 'SB'
            elif c+1 == 3:
                Rhythm = 'SR'
            else:
                Rhythm = 'GSVT'
            result.append([Rhythm,acc,recall,precision,f1_score,specificity])
        p_macro,r_macro,f_macro,support_macro = precision_recall_fscore_support(y,y_pred,average='macro')
        p_micro,r_micro,f_micro,support_micro = precision_recall_fscore_support(y,y_pred,average='micro')
        p_weighted,r_weighted,f_weighted,support_weighted = precision_recall_fscore_support(y,y_pred,average='weighted')

        s_marco = np.mean(specificities)
        tn_total = np.sum([np.sum(np.delete(np.delete(cm, i, axis=0), i, axis=1)) for i in range(n_classes)])
        fp_total = np.sum([np.sum(cm[:, i]) - cm[i, i] for i in range(n_classes)])
        s_mirco = tn_total / (tn_total + fp_total)
        s_weighted = np.sum(weights * specificities)

        result.append(['macro avg',None,f_macro,p_macro,r_macro,s_marco])
        result.append(['micro avg',None,f_micro,p_micro,r_micro,s_mirco])
        result.append(['weighted avg',None,f_weighted,p_weighted,r_weighted,s_weighted, d_train, time])
        return result
    evaluation_test = evaluation_test(y_test, result_test, res_etime)
    df_evaluation_test = pd.DataFrame(data=evaluation_test,columns=["Rhythm Group","ACC","F1-score","Precision","Recall","specificity","Feature", "Time"])
    var_text = str(var).replace('.','_')
    df_evaluation_test.to_csv(f"./Result_full/XGB_{var_text}.csv")
        

Fitting 3 folds for each of 8 candidates, totalling 24 fits


0.99 -> 138
0.98 -> 116
0.96 -> 90
0.95 -> 81
0.90 -> 51